In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent

sys.path.append(str(ROOT))

# Regression

## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import time
from sklearn.metrics import mean_squared_error
from dataset import CaliforniaHousingDataset as Dataset
from pipeline import get_pipeline
from eda.eda import EDA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
plt.rcParams['figure.figsize'] = (12, 8)

# Linear regression
from Part1_Regression.linear_regression.diagnostics import GaussMarkovDiagnostics
from Part1_Regression.linear_regression.ols import OLS
from Part1_Regression.linear_regression.wls import WLS

# Regularization and features selection
from Part1_Regression.regularization.ridge_regression import RidgeRegression, RidgeRegressionCV
from Part1_Regression.regularization.lasso_regression import LassoRegression, LassoRegressionCV
from Part1_Regression.regularization.elastic_net import ElasticNet, ElasticNetCV
from Part1_Regression.features_selection.features_selection import \
    forward_selection, backward_elimination, lasso_selection

# Nonlinear basis
from Part1_Regression.nonlinear_basis.polynomial import PolynomialBasis
from Part1_Regression.nonlinear_basis.fourier import FourierBasis
from Part1_Regression.nonlinear_basis.rbf import RBF

# Advanced
from Part1_Regression.advanced.gpr import GPR
from Part1_Regression.advanced.irls import IRLS
from Part1_Regression.advanced.bayes_reg import BayesianLinearRegression
from Part1_Regression.advanced.bayesian_em_vs_cv import compare_em_vs_cv

# Evaluation
from Part1_Regression.evaluation.visualizer import Visualizer
from Part1_Regression.evaluation.evaluator import Evaluator

## Dataset

### Missing values

In [ ]:
    d = Dataset()
    eda = EDA(d)

    print(eda.missing_values())


    # eda.plot_target_distribution()
    # eda.plot_scatter_features_target()
    # eda.plot_corrrelation_matrix()
    # eda.plot_outliers()
    # plt.show()

### Descriptive statistics

In [ ]:
print(eda.descriptive_stats())

### Target distribution

In [ ]:
eda.plot_target_distribution()
plt.show()

### Scatter

In [ ]:
eda.plot_scatter_features_target()
plt.show()

### Correlation matrix

In [ ]:
eda.plot_corrrelation_matrix()
plt.show()

### Outliers

In [ ]:
eda.plot_outliers()
plt.show()

### Preprocessing

In [ ]:
d.split()

## Models

### Linear regression

#### Gauss-Markov hypothesis

In [ ]:
model = get_pipeline(OLS())
model.fit(d.X_train, d.y_train)
y_pred_train = model.predict(d.X_train)
diag = GaussMarkovDiagnostics()

##### Residuals plot

In [ ]:
residuals_train = d.y_train - y_pred_train
diag.plot_residuals(d.y_train, y_pred_train)

plt.show()

##### QQ-plot

In [ ]:
diag.plot_qq(residuals_train)
plt.show()

##### Breusch–Pagan test

In [ ]:
result = diag.breusch_pagan_test(d.X_train, residuals_train)
print(result)

if result["LM p-value"] < 0.05:
    aux = get_pipeline(OLS())
    aux.fit(d.X_train, np.square(residuals_train))

    sigma2_hat = aux.predict(d.X_train)
    sigma2_hat = np.clip(sigma2_hat, 1e-6, None)

    weights = 1.0 / sigma2_hat

    wls = get_pipeline(WLS(weights=weights))
    wls.fit(d.X_train, d.y_train)

    y_pred_train = wls.predict(d.X_train)
    residuals_train = d.y_train - y_pred_train
    weighted_residuals = np.sqrt(weights) * residuals_train

    diag.plot_residuals_direct(y_pred_train, weighted_residuals)
    plt.show()
    diag.plot_qq(weighted_residuals)
    plt.show()

#### Regularization and features selecton

##### Ridge Regression

In [ ]:
lambdas = np.logspace(-10, 1, 12)

In [ ]:
model = get_pipeline(RidgeRegressionCV(alphas=lambdas, cv=10))
model.fit(d.X_train, d.y_train)

In [ ]:
print(model.named_steps["predictor"].best_alpha_)

In [ ]:
model.named_steps["predictor"].plot_regularization_path(title="Rigde Regularization Path")

##### Lasso Regression

In [ ]:
lambdas = np.logspace(-10, 1, 12)

In [ ]:
model = get_pipeline(LassoRegressionCV(alphas=lambdas, cv=10))
model.fit(d.X_train, d.y_train)

In [ ]:
print(model.named_steps["predictor"].best_alpha_)

In [ ]:
model.named_steps["predictor"].plot_regularization_path(title="Rigde Regularization Path")

##### Elastic Net

In [ ]:
lambda_1s = np.logspace(-10, 1, 12)
lambda_2s = np.logspace(-10, 1, 12)

In [ ]:
model = get_pipeline(ElasticNetCV(alpha_1s=lambda_1s, alpha_2s=lambda_2s, cv=10))
model.fit(d.X_train, d.y_train)

In [ ]:
model.named_steps["predictor"].plot_optimal_region(title="Elastic Net Optimal Region")

##### Features Selection

In [ ]:
k_features = 5

In [ ]:
model = get_pipeline(LassoRegression())

###### Forward

In [ ]:
forward_selection(d.X_train, d.y_train, model, k_features, d.feature_names)

###### Backward

In [ ]:
backward_elimination(d.X_train, d.y_train, model, k_features, d.feature_names)

###### Lasso

In [ ]:
lambdas = np.logspace(-10, 1, 12)

In [ ]:
lasso_selection(d.X_train, d.y_train, d.feature_names, alphas=lambdas, cv=10)

#### Non-linear basis and Ablation study

##### Polynomial

### Advanced

#### Bayesian Regression


In [ ]:
# ===== MODEL =====
X_train, y_train = d.X_train, d.y_train
X_test, y_test = d.X_test, d.y_test
model = BayesianLinearRegression(alpha=1.0, beta=10.0)
model.fit(X_train, y_train)


In [ ]:
 # ===== POSTERIOR =====
m_N, S_N = model.get_posterior()
print("Posterior mean shape:", m_N.shape)
print("Posterior covariance shape:", S_N.shape)

In [ ]:
 # ===== PREDICT =====
y_mean, y_std = model.predict_dist(X_test)
mse = np.mean((y_test - y_mean) ** 2)
print(f"MSE = {mse:.6f}")

In [ ]:
# ===== PLOT (1D visualization) =====
X_plot = X_test[:, 0]
idx = np.argsort(X_plot)

plt.figure(figsize=(8, 5))

# data test
plt.scatter(X_plot, y_test, s=10, label="Test data")

# mean prediction
plt.plot(X_plot[idx], y_mean[idx], label="Mean")

# vùng bất định ±2σ
plt.fill_between(
    X_plot[idx],
    y_mean[idx] - 2*y_std[idx],
    y_mean[idx] + 2*y_std[idx],
    alpha=0.3,
    label="±2σ"
)
plt.xlabel("Median Income (MedInc) - Feature 0")
plt.ylabel("House Price (Target)")
plt.legend()
plt.title("Bayesian Linear Regression (Uncertainty)")
plt.show()


In [ ]:
#Optimazation alpha , beta using Evidence Maximization to compare with CV 
start = time.time()

model_em = BayesianLinearRegression()
model_em.evidence_maximization(X_train, y_train)

y_pred_em = model_em.predict(X_test)

em_time = time.time() - start
em_mse = mean_squared_error(y_test, y_pred_em)

#  CV 
param_grid = {
    "alpha": [0.1, 1, 10],
    "beta": [1, 10, 100]
}

start = time.time()

grid = GridSearchCV(
    BayesianLinearRegression(),
    param_grid,
    cv=3
)
grid.fit(X_train, y_train)

y_pred_cv = grid.predict(X_test)

cv_time = time.time() - start
cv_mse = mean_squared_error(y_test, y_pred_cv)

# ================= RESULT =================
print("\n===== EVIDENCE MAXIMIZATION =====")
print("alpha:", model_em.alpha)
print("beta:", model_em.beta)
print("MSE:", em_mse)
print("Time:", em_time)

print("\n===== CROSS VALIDATION =====")
print("Best params:", grid.best_params_)
print("MSE:", cv_mse)
print("Time:", cv_time)

print("\n===== COMPARISON =====")
print(f"EM faster than CV? {em_time < cv_time}")
print(f"EM better MSE than CV? {em_mse < cv_mse}")

#### Gaussian Process Regression

In [ ]:
rng = np.random.default_rng(42)
idx = rng.choice(len(d.X_train), 2000, replace=False)
X_sub = d.X_train[idx]
y_sub = d.y_train[idx]

model = get_pipeline(GPR())
model.fit(X_sub, y_sub)

y_pred, y_std = model.predict(d.X_test)
y_true = d.y_test
print("min std:", y_std.min())
print("max std:", y_std.max())

X_test = d.X_test

In [ ]:
mean_squared_error(y_true, y_pred)

In [ ]:
feature_names = getattr(d, "feature_names", [f"Feature {i}" for i in range(X_test.shape[1])])

n_features = X_test.shape[1]

for i in range(n_features):
    x_plot = X_test[:, i]
    idx = np.argsort(x_plot)

    plt.subplot(3, 3, i+1)

    # Prediction mean
    plt.plot(x_plot[idx], y_pred[idx], label="Pred", linewidth=1)

    # Uncertainty band
    plt.fill_between(
        x_plot[idx],
        y_pred[idx] - 1.96*y_std[idx],
        y_pred[idx] + 1.96*y_std[idx],
        # color="red",
        alpha=0.2
    )

    # True points
    plt.scatter(x_plot, y_true, s=5, alpha=0.3)

    plt.title(feature_names[i])
    plt.xlabel("Feature value")
    plt.ylabel("Price")
plt.gcf().suptitle("Posterior predictive with error bars")
plt.tight_layout()
plt.show()

#### Robust Regression

In [ ]:
huber = get_pipeline(IRLS(loss="huber"))
huber.fit(d.X_train, d.y_train)
y_pred_huber = huber.predict(d.X_test)

student = get_pipeline(IRLS(loss="student-t"))
student.fit(d.X_train, d.y_train)
y_pred_student = student.predict(d.X_test)

model_ols = get_pipeline(OLS())
model_ols.fit(d.X_train, d.y_train)
y_pred_ols = model_ols.predict(d.X_test)

y_true = d.y_test

print("=== Overall MSE ===")
print(f"OLS: {mean_squared_error(y_true, y_pred_ols):.4f}")
print(f"Huber IRLS: {mean_squared_error(y_true, y_pred_huber):.4f}")
print(f"Student-t IRLS: {mean_squared_error(y_true, y_pred_student):.4f}")

In [ ]:
# ===== TRAIN CLEAN =====
model_ols.fit(d.X_train, d.y_train)
huber.fit(d.X_train, d.y_train)
student.fit(d.X_train, d.y_train)

y_ols_clean = model_ols.predict(d.X_test)
y_huber_clean = huber.predict(d.X_test)
y_student_clean = student.predict(d.X_test)

# ===== ADD OUTLIER =====
y_train_noisy = d.y_train.copy()

idx = np.random.choice(len(d.y_train), int(0.1 * len(d.y_train)), replace=False)
y_train_noisy[idx] += 5 * np.std(d.y_train)

# ===== TRAIN NOISY =====
model_ols.fit(d.X_train, y_train_noisy)
huber.fit(d.X_train, y_train_noisy)
student.fit(d.X_train, y_train_noisy)

y_ols_noisy = model_ols.predict(d.X_test)
y_huber_noisy = huber.predict(d.X_test)
y_student_noisy = student.predict(d.X_test)

# ===== SENSITIVITY =====
print("\n=== Sensitivity ===")
print("OLS:", np.mean(np.abs(y_ols_clean - y_ols_noisy)))
print("Huber:", np.mean(np.abs(y_huber_clean - y_huber_noisy)))
print("Student:", np.mean(np.abs(y_student_clean - y_student_noisy)))

### Models evaluation

In [ ]:
models = {
    "OLS": get_pipeline(OLS()),
    "IRLS (Huber)": get_pipeline(IRLS(loss="huber")),
    "IRLS (Student-t)": get_pipeline(IRLS(loss="student-t")),
    # "Polynomial": get_pipeline(
    #     LinearRegression(),
    #     PolynomialBasis(degree=2) 
    # ),
    # "RBF": get_pipeline(LinearRegression(),
    #     RBF(n_centers=10, gamma=0.1)),
    # "Fourier": get_pipeline(
    #     LinearRegression(),
    #     FourierBasis(n_terms=5) 
    # )
}
evaluator = Evaluator()

result = evaluator.compare_models_cv(models, d.X_train, d.y_train)

print(result)

result = evaluator.compare_models_test(models, d.X_train, d.y_train,
                                       d.X_test, d.y_test)
print(result)

stat_result = evaluator.compare_models_statistical(
    models,
    d.X_train,
    d.y_train
)

print(stat_result)